# 🚗 YOLO API - PKTJ Kinerja Ruas Jalan

Notebook ini menjalankan YOLO detection API untuk proyek **Kinerja Ruas Jalan**.

## Cara Pakai:
1. Jalankan semua cell dari atas ke bawah
2. Copy URL ngrok yang muncul di cell terakhir
3. Paste ke file `backend/.env` → `YOLO_API_URL=https://xxxxx.ngrok-free.app`
4. Restart backend Node.js

**Runtime:** Pilih `T4 GPU` di Runtime → Change runtime type

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics fastapi uvicorn python-multipart pyngrok cloudinary python-dotenv nest_asyncio httpx
print('✅ Dependencies installed')

## Cell 2 — Mount Google Drive (untuk dataset & model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

## Cell 3 — Setup Folder & Salin Dataset dari Drive

> **Sebelum jalankan:** Upload folder dataset ke Google Drive terlebih dahulu.
> Struktur yang diharapkan di Drive:
> ```
> MyDrive/pktj/
> ├── models/best.pt
> └── data/
>     ├── images/train/*.jpg
>     ├── images/val/*.jpg
>     ├── labels/train/*.txt
>     └── labels/val/*.txt
> ```

In [ ]:
import os, shutil

# ===== SESUAIKAN PATH INI DENGAN LOKASI FILE DI GOOGLE DRIVE =====
DRIVE_PKTJ_PATH = '/content/drive/MyDrive/pktj'
# =================================================================

BASE_DIR = '/content/pktj'
MODEL_DIR = f'{BASE_DIR}/models'
DATA_DIR = f'{BASE_DIR}/data'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(f'{DATA_DIR}/images/train', exist_ok=True)
os.makedirs(f'{DATA_DIR}/images/val', exist_ok=True)
os.makedirs(f'{DATA_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{DATA_DIR}/labels/val', exist_ok=True)

# Salin model
model_src = f'{DRIVE_PKTJ_PATH}/models/best.pt'
if os.path.exists(model_src):
    shutil.copy(model_src, f'{MODEL_DIR}/best.pt')
    print(f'✅ Model disalin: {MODEL_DIR}/best.pt ({os.path.getsize(MODEL_DIR+"/best.pt")/1e6:.1f} MB)')
else:
    print(f'⚠️ Model tidak ditemukan di {model_src}')
    print('   Download YOLOv8n sebagai fallback...')
    from ultralytics import YOLO
    model = YOLO('yolov8n.pt')
    shutil.copy('yolov8n.pt', f'{MODEL_DIR}/best.pt')
    print('✅ YOLOv8n digunakan sebagai fallback')

# Salin dataset
data_src = f'{DRIVE_PKTJ_PATH}/data'
if os.path.exists(data_src):
    for split in ['train', 'val']:
        for dtype in ['images', 'labels']:
            src = f'{data_src}/{dtype}/{split}'
            dst = f'{DATA_DIR}/{dtype}/{split}'
            if os.path.exists(src):
                files = os.listdir(src)
                for f in files:
                    shutil.copy(f'{src}/{f}', f'{dst}/{f}')
                print(f'✅ {dtype}/{split}: {len(files)} files disalin')
else:
    print(f'⚠️ Dataset tidak ditemukan di {data_src}')
    print('   Lanjutkan ke training jika ingin train ulang, atau skip ke Cell 5 jika hanya ingin inference')

print('\n📁 Struktur folder:')
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{subindent}{file}')
        if len(files) > 3:
            print(f'{subindent}... ({len(files)} files total)')

## Cell 4 — (Opsional) Training Ulang Model

> **Skip cell ini** jika `best.pt` sudah ada dan tidak perlu training ulang.

In [ ]:
# Buat data.yaml untuk training
data_yaml = f"""
path: {DATA_DIR}
train: images/train
val: images/val

nc: 3
names:
  0: mobil
  1: bus
  2: truk
"""

with open(f'{BASE_DIR}/data.yaml', 'w') as f:
    f.write(data_yaml)

print('data.yaml dibuat:')
print(data_yaml)

# Cek jumlah data
train_imgs = len(os.listdir(f'{DATA_DIR}/images/train'))
val_imgs = len(os.listdir(f'{DATA_DIR}/images/val'))
print(f'\nDataset: {train_imgs} train, {val_imgs} val images')

In [ ]:
# ⚠️ JALANKAN HANYA JIKA INGIN TRAINING ULANG
# Training membutuhkan waktu 30-60 menit tergantung dataset

from ultralytics import YOLO

# Load base model
model = YOLO('yolov8n.pt')  # atau yolov8s.pt untuk lebih akurat

# Training
results = model.train(
    data=f'{BASE_DIR}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # 0 = GPU
    project=BASE_DIR,
    name='train',
    exist_ok=True,
    patience=10,        # Early stopping
    save=True,
    plots=True
)

# Salin model terbaik
best_model = f'{BASE_DIR}/train/weights/best.pt'
if os.path.exists(best_model):
    shutil.copy(best_model, f'{MODEL_DIR}/best.pt')
    print(f'✅ Model terbaik disalin ke {MODEL_DIR}/best.pt')

    # Simpan juga ke Drive
    os.makedirs(f'{DRIVE_PKTJ_PATH}/models', exist_ok=True)
    shutil.copy(best_model, f'{DRIVE_PKTJ_PATH}/models/best.pt')
    print(f'✅ Model disimpan ke Google Drive')

## Cell 5 — Konfigurasi Cloudinary & Ngrok

In [ ]:
# ===== ISI KONFIGURASI INI =====
CLOUDINARY_CLOUD_NAME = 'dgiv24lgt'          # dari .env.production frontend
CLOUDINARY_API_KEY    = 'ISI_API_KEY_ANDA'   # dari dashboard Cloudinary
CLOUDINARY_API_SECRET = 'ISI_API_SECRET_ANDA'# dari dashboard Cloudinary

NGROK_AUTH_TOKEN = 'ISI_NGROK_TOKEN_ANDA'    # dari https://dashboard.ngrok.com/get-started/your-authtoken
# ================================

import os
os.environ['CLOUDINARY_CLOUD_NAME'] = CLOUDINARY_CLOUD_NAME
os.environ['CLOUDINARY_API_KEY']    = CLOUDINARY_API_KEY
os.environ['CLOUDINARY_API_SECRET'] = CLOUDINARY_API_SECRET

print('✅ Konfigurasi disimpan')
print(f'   Cloudinary: {CLOUDINARY_CLOUD_NAME}')
print(f'   Ngrok token: {NGROK_AUTH_TOKEN[:8]}...')

## Cell 6 — Buat FastAPI Server

In [ ]:
api_code = '''
import os, gc, asyncio, tempfile, json
import cv2
import numpy as np
from fastapi import FastAPI, UploadFile, File, BackgroundTasks, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO
import cloudinary
import cloudinary.uploader

# ===== CONFIG =====
MODEL_PATH  = "/content/pktj/models/best.pt"
JOBS_DIR    = "/content/pktj/jobs"
CONF_THRESH = 0.25
IOU_THRESH  = 0.4
MAX_DISAP   = 15
MAX_DIST    = 120
MIN_FRAMES  = 2
CLASS_MAP   = {0: "mobil", 1: "bus", 2: "truk"}

os.makedirs(JOBS_DIR, exist_ok=True)

cloudinary.config(
    cloud_name=os.getenv("CLOUDINARY_CLOUD_NAME"),
    api_key=os.getenv("CLOUDINARY_API_KEY"),
    api_secret=os.getenv("CLOUDINARY_API_SECRET"),
)

model = YOLO(MODEL_PATH)
jobs  = {}

def save_job(job_id):
    try:
        with open(f"{JOBS_DIR}/{job_id}.json", "w") as f:
            json.dump(jobs[job_id], f)
    except Exception as e:
        print(f"Save job error: {e}")

def load_job(job_id):
    try:
        path = f"{JOBS_DIR}/{job_id}.json"
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
    except:
        pass
    return None

def get_lane(cx, fw): return "kiri" if cx < fw // 2 else "kanan"

def calc_iou(b1, b2):
    xi1,yi1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    xi2,yi2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    if xi2<xi1 or yi2<yi1: return 0.0
    inter = (xi2-xi1)*(yi2-yi1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter/union if union>0 else 0

def euc(p1,p2): return np.sqrt((p1[0]-p2[0])**2+(p1[1]-p2[1])**2)

def check_crossing(hist, line_y, offset=40):
    if len(hist)<2: return False
    py,cy = hist[-2],hist[-1]
    return (py<=line_y and cy>line_y) or (py<line_y+offset and cy>=line_y+offset)

def update_tracking(objs, nid, dets, fw, counters, total):
    if not dets:
        for oid in list(objs):
            objs[oid]["dis"] += 1
            mx = 5 if objs[oid]["counted"] else MAX_DISAP
            if objs[oid]["dis"] > mx: del objs[oid]
        return objs, nid, total

    # NMS manual
    filtered = []
    for d in sorted(dets, key=lambda x: x["conf"], reverse=True):
        if all(calc_iou(d["bbox"], e["bbox"]) <= IOU_THRESH for e in filtered):
            filtered.append(d)
    dets = filtered

    if not objs:
        for d in dets:
            lane = get_lane(d["c"][0], fw)
            objs[nid] = {"c":d["c"],"bbox":d["bbox"],"dis":0,"counted":False,
                         "cls":d["cls"],"lane":lane,"hist":[d["c"][1]],"fc":1}
            nid += 1
        return objs, nid, total

    oids = list(objs)
    ocs  = [objs[o]["c"] for o in oids]
    ics  = [d["c"] for d in dets]
    dist = np.array([[euc(a,b) for b in ics] for a in ocs])

    used_r, used_c = set(), set()
    rows = dist.min(axis=1).argsort()
    cols = dist.argmin(axis=1)[rows]

    for r,c in zip(rows,cols):
        if r in used_r or c in used_c or dist[r,c]>MAX_DIST: continue
        oid, d = oids[r], dets[c]
        if not objs[oid]["counted"]: objs[oid]["cls"] = d["cls"]
        objs[oid].update({"c":d["c"],"bbox":d["bbox"],"dis":0})
        objs[oid]["hist"].append(d["c"][1])
        if len(objs[oid]["hist"])>15: objs[oid]["hist"].pop(0)
        objs[oid]["fc"] += 1
        objs[oid]["lane"] = get_lane(d["c"][0], fw)
        if not objs[oid]["counted"] and objs[oid]["fc"]>=MIN_FRAMES:
            if check_crossing(objs[oid]["hist"], fw//2):
                objs[oid]["counted"] = True
                ln = objs[oid]["lane"]
                cls = objs[oid]["cls"]
                counters[ln]["total"] += 1
                counters[ln][cls] += 1
                total += 1
        used_r.add(r); used_c.add(c)

    for r in set(range(len(oids)))-used_r:
        oid = oids[r]
        objs[oid]["dis"] += 1
        if objs[oid]["dis"] > (5 if objs[oid]["counted"] else MAX_DISAP): del objs[oid]

    for c in set(range(len(dets)))-used_c:
        d = dets[c]
        lane = get_lane(d["c"][0], fw)
        objs[nid] = {"c":d["c"],"bbox":d["bbox"],"dis":0,"counted":False,
                     "cls":d["cls"],"lane":lane,"hist":[d["c"][1]],"fc":1}
        nid += 1

    return objs, nid, total

async def process_video_task(job_id: str, video_url: str):
    objs, nid, total = {}, 0, 0
    counters = {"kiri":{"total":0,"mobil":0,"bus":0,"truk":0},
                "kanan":{"total":0,"mobil":0,"bus":0,"truk":0}}

    jobs[job_id].update({"status":"processing","progress":0})
    save_job(job_id)

    cap = cv2.VideoCapture(video_url)
    if not cap.isOpened():
        jobs[job_id].update({"status":"failed","error":"Tidak bisa buka video"})
        save_job(job_id); return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    frame_count  = 0
    PROC_W       = 640
    line_y       = None

    fourcc  = cv2.VideoWriter_fourcc(*"mp4v")
    out_path= f"/tmp/output_{job_id}.mp4"
    writer  = None

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        h, w, _ = frame.shape
        if line_y is None: line_y = h // 2

        scale = PROC_W / w
        small = cv2.resize(frame, (PROC_W, int(h * scale)))
        res   = model.predict(small, device=0, conf=CONF_THRESH, verbose=False, max_det=50)

        dets = []
        if res[0].boxes is not None:
            for box, cid, cf in zip(
                res[0].boxes.xyxy.cpu().numpy(),
                res[0].boxes.cls.cpu().numpy().astype(int),
                res[0].boxes.conf.cpu().numpy()
            ):
                x1,y1,x2,y2 = box/scale
                dets.append({"bbox":[x1,y1,x2,y2],
                              "c":(int((x1+x2)/2),int((y1+y2)/2)),
                              "cls":CLASS_MAP.get(cid,"mobil"),"conf":float(cf)})

        cv2.line(frame,(0,line_y),(w,line_y),(0,0,255),2)
        if writer is None: writer = cv2.VideoWriter(out_path, fourcc, 20.0, (w,h))
        writer.write(frame)

        del frame, small, res
        objs, nid, total = update_tracking(objs, nid, dets, w, counters, total)

        prog = int(frame_count/total_frames*100)
        jobs[job_id].update({"progress":prog,"total":total,
            "kiri":counters["kiri"]["total"],"kanan":counters["kanan"]["total"]})
        if prog % 10 == 0: save_job(job_id); gc.collect()
        await asyncio.sleep(0)

    cap.release()
    if writer: writer.release()

    # Re-encode
    import subprocess
    enc_path = out_path.replace(".mp4","_enc.mp4")
    subprocess.run(["ffmpeg","-y","-i",out_path,"-c:v","libx264","-movflags","+faststart",enc_path],
                   capture_output=True)
    upload_path = enc_path if os.path.exists(enc_path) else out_path

    # Upload Cloudinary
    out_url = None
    try:
        r = cloudinary.uploader.upload(upload_path, resource_type="video", folder="output_videos")
        out_url = r["secure_url"]
    except Exception as e:
        print(f"Cloudinary upload error: {e}")

    for p in [out_path, enc_path]:
        if os.path.exists(p): os.remove(p)
    gc.collect()

    jobs[job_id].update({
        "status":"completed","progress":100,"completed":True,
        "outputVideoUrl":out_url,"vehicle_count":total,
        "frames_processed":frame_count,
        "lane":{
            "kiri":{"total":counters["kiri"]["total"],"mobil":counters["kiri"]["mobil"],
                    "bus":counters["kiri"]["bus"],"truk":counters["kiri"]["truk"]},
            "kanan":{"total":counters["kanan"]["total"],"mobil":counters["kanan"]["mobil"],
                     "bus":counters["kanan"]["bus"],"truk":counters["kanan"]["truk"]}
        }
    })
    save_job(job_id)
    print(f"✅ Job {job_id} selesai: {total} kendaraan dari {frame_count} frames")

# ===== FASTAPI APP =====
app = FastAPI(title="YOLO PKTJ API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def root(): return {"status":"YOLO API berjalan","model":"best.pt","classes":["mobil","bus","truk"]}

@app.get("/ping")
def ping(): return {"pong":True}

@app.post("/detect")
async def detect(file: UploadFile = File(...), background_tasks: BackgroundTasks = None):
    suffix = os.path.splitext(file.filename)[1]
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read()); tmp_path = tmp.name
    try:
        r = cloudinary.uploader.upload(tmp_path, resource_type="video", folder="input_videos")
        video_url = r["secure_url"]
    finally:
        os.remove(tmp_path)
    import time
    job_id = str(int(time.time() * 1000))
    jobs[job_id] = {"status":"queued","progress":0}
    background_tasks.add_task(process_video_task, job_id, video_url)
    return {"job_id":job_id,"video_url":video_url}

@app.get("/result/{job_id}")
def get_result(job_id: str):
    job = jobs.get(job_id) or load_job(job_id)
    if not job: return JSONResponse({"status":"pending","progress":0})
    return JSONResponse({
        "job_id":job_id,
        "status":"completed" if job.get("completed") else job.get("status","processing"),
        "progress":job.get("progress",0),
        "vehicle_count":job.get("vehicle_count",0),
        "frames_processed":job.get("frames_processed",0),
        "lane":job.get("lane",{"kiri":{"total":0,"mobil":0,"bus":0,"truk":0},"kanan":{"total":0,"mobil":0,"bus":0,"truk":0}}),
        "outputVideoUrl":job.get("outputVideoUrl")
    })
'''

with open('/content/pktj/api.py', 'w') as f:
    f.write(api_code)

print('✅ api.py dibuat di /content/pktj/api.py')

## Cell 7 — Jalankan API + Ngrok

In [ ]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf

nest_asyncio.apply()

# Setup ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()  # Matikan tunnel lama jika ada

# Mulai tunnel
tunnel = ngrok.connect(8000, "http")
NGROK_URL = tunnel.public_url

print('=' * 60)
print('🚀 YOLO API siap!')
print('=' * 60)
print(f'📡 URL Ngrok   : {NGROK_URL}')
print(f'🔍 Test ping   : {NGROK_URL}/ping')
print()
print('📋 LANGKAH SELANJUTNYA:')
print(f'  1. Buka file backend/.env')
print(f'  2. Ubah YOLO_API_URL={NGROK_URL}')
print(f'  3. Restart backend Node.js')
print('=' * 60)

# Jalankan FastAPI
import sys
sys.path.insert(0, '/content/pktj')

config = uvicorn.Config(
    'api:app',
    host='0.0.0.0',
    port=8000,
    log_level='warning',
    app_dir='/content/pktj'
)
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print('\n⏳ Server berjalan... (cell ini tetap aktif selama API berjalan)')
print('   Tekan tombol Stop (■) untuk menghentikan.')

## Cell 8 — Test API (Opsional)

In [ ]:
import httpx

# Test ping
r = httpx.get(f'{NGROK_URL}/ping')
print('Ping:', r.json())

# Test root
r = httpx.get(f'{NGROK_URL}/')
print('Root:', r.json())

## Cell 9 — Simpan Model ke Drive (setelah training)

> Jalankan ini setelah training untuk backup model ke Google Drive

In [ ]:
import shutil, os

os.makedirs(f'{DRIVE_PKTJ_PATH}/models', exist_ok=True)
shutil.copy('/content/pktj/models/best.pt', f'{DRIVE_PKTJ_PATH}/models/best.pt')

size = os.path.getsize(f'{DRIVE_PKTJ_PATH}/models/best.pt') / 1e6
print(f'✅ Model disimpan ke Google Drive ({size:.1f} MB)')
print(f'   Path: {DRIVE_PKTJ_PATH}/models/best.pt')